# Objectives

What the solver *minimizes*. The objective is a generalized cost:

```
duration_weight x duration + distance_weight x distance
+ fixed vehicle costs + soft-constraint penalties + waiting price
- profit_weight x collected prizes
```

Everything is expressed in one unit — by default, the units of your duration
matrix. Docs: Multi-Objective and Prize-Collecting feature pages.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt

from tqrouting import TQRouter, VRPInstance, Customer, VehicleType, Location, TimeWindow
from tqrouting.matrix import EuclideanMatrixProvider

data_dir = Path("data")

# Uses the TQROUTING_LICENSE_KEY environment variable by default.
# Set it before running: export TQROUTING_LICENSE_KEY="key/..."
router = TQRouter()

## 1. Duration vs distance

By default only duration matters. `distance_weight` prices distance in:
with second/meter matrices, `0.1` means 1 km costs as much as 100 s.
A fast-but-long highway and a slow-but-short back road make the trade-off visible.

In [2]:
# Two customers. The depot->highway leg is FAST but LONG (a motorway detour);
# the depot->backroad leg is slow but short. Which customer to serve first?
#   serve highway first:  660 s of driving over 12000 m
#   serve backroad first: 960 s of driving over  7000 m
durations = [   # seconds:  D       highway  backroad
    [0, 300, 600],          # D
    [300, 0, 360],          # highway
    [600, 360, 0],          # backroad
]
distances = [   # meters
    [0, 9000, 4000],
    [9000, 0, 3000],
    [4000, 3000, 0],
]
inst = VRPInstance(
    customers=[
        Customer(id="highway", location=Location(matrix_id=1)),
        Customer(id="backroad", location=Location(matrix_id=2)),
    ],
    fleet=[VehicleType(start_location=Location(matrix_id=0), n_vehicles=1)],
    duration_matrix=durations,
    distance_matrix=distances,
)

for w in (0.0, 0.5):
    sol = router.solve(inst, distance_weight=w, time_limit=2)
    order = [v.customer_id for v in sol.routes[0].customer_sequence]
    print(f"distance_weight={w}: visit order {order}, "
          f"duration {sol.total_route_duration:.0f}s, "
          f"distance {sol.total_driving_distance:.0f}m")

[tqrouting] Solving VRP instance (2 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 1 routes, duration 660.0, 0 unvisited).


[tqrouting] Solving VRP instance (2 customers, 1 vehicle types)...


distance_weight=0.0: visit order ['highway', 'backroad'], duration 660s, distance 12000m


[tqrouting] Solver finished in 2.0s (status=Feasible, 1 routes, duration 960.0, 0 unvisited).


distance_weight=0.5: visit order ['backroad', 'highway'], duration 960s, distance 7000m


Pure distance minimization: set `duration_weight=0` (the classic distance-CVRP
objective). Time windows and duration limits are still *enforced* — weights
change what is minimized, never what is feasible.

In [3]:
sol_dist = router.solve(inst, duration_weight=0.0, distance_weight=1.0, time_limit=2)
print(f"Pure distance: {sol_dist.total_driving_distance:.0f}m, "
      f"duration {sol_dist.total_route_duration:.0f}s")

[tqrouting] Solving VRP instance (2 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 1 routes, duration 960.0, 0 unvisited).


Pure distance: 7000m, duration 960s


## 2. Prize-collecting (Team Orienteering)

Customers carry a `profit` — a prize in objective units. A customer is visited
only when its prize outweighs the detour to reach it. Prizes must be
comparable to travel costs: here durations are seconds, so `profit=250`
cannot pay for a 300 s detour.

In [4]:
inst_pc = VRPInstance(
    customers=[
        Customer(id="big", location=Location(x=600, y=0), profit=2400),
        Customer(id="good", location=Location(x=0, y=600), profit=1500),
        Customer(id="ok", location=Location(x=-500, y=0), profit=900),
        Customer(id="tiny", location=Location(x=0, y=-700), profit=250),
    ],
    fleet=[VehicleType(start_location=Location(x=0, y=0), n_vehicles=1)],
)
inst_pc.compute_duration_matrix(EuclideanMatrixProvider())
sol_pc = router.solve(inst_pc, time_limit=2)
visited = [v.customer_id for r in sol_pc.routes for v in r.customer_sequence]
print(f"Visited: {visited}")
print(f"Skipped: {sol_pc.unvisited_customers}")
print(f"Collected profit: {sol_pc.total_profit:.0f}")

[tqrouting] Computing duration matrix for 5 unique locations...


[tqrouting] Duration matrix ready (5×5, 0.0s).


[tqrouting] Solving VRP instance (4 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 1 routes, duration 2129.6, 1 unvisited).


Visited: ['ok', 'good', 'big']
Skipped: ['tiny']
Collected profit: 4800


A larger instance: `top_100.json`, 100 customers with prizes between 1 and 30,
two vehicles limited to 55 time units each, on a map with arcs up to ~40
units. The prizes are small next to travel costs, so `profit_weight` scales
them up to comparable magnitude.

In [5]:
instance_top = VRPInstance.from_json(data_dir / "top_100.json")
total = sum(c.profit for c in instance_top.customers if c.profit)
sol_top = router.solve(instance_top, time_limit=10, profit_weight=5)
print(f"Collected {sol_top.total_profit:.0f} of {total:.0f} available profit "
      f"with {len(sol_top.routes)} routes; "
      f"skipped {len(sol_top.unvisited_customers or [])} customers")

[tqrouting] Solving VRP instance (100 customers, 1 vehicle types)...


[tqrouting] Solver finished in 10.0s (status=Feasible, 2 routes, duration 109.9, 54 unvisited).


Collected 683 of 1594 available profit with 2 routes; skipped 54 customers


## 3. Fixed vehicle cost — fleet minimization

`fixed_vehicle_cost` charges each used vehicle, in objective units.
Two clusters, cheap to serve with two vehicles — until each vehicle
costs the equivalent of 5000 s.

In [6]:
def two_cluster_instance(fixed_cost):
    return VRPInstance(
        customers=[
            Customer(location=Location(x=2000, y=100)),
            Customer(location=Location(x=2100, y=0)),
            Customer(location=Location(x=-2000, y=100)),
            Customer(location=Location(x=-2100, y=0)),
        ],
        fleet=[VehicleType(start_location=Location(x=0, y=0), n_vehicles=2,
                           fixed_vehicle_cost=fixed_cost)],
    )

for cost in (0, 5000):
    inst_fc = two_cluster_instance(cost)
    inst_fc.compute_duration_matrix(EuclideanMatrixProvider())
    sol_fc = router.solve(inst_fc, time_limit=2)
    print(f"fixed_vehicle_cost={cost}: {len(sol_fc.routes)} vehicle(s), "
          f"total duration {sol_fc.total_route_duration:.0f}s")

[tqrouting] Computing duration matrix for 5 unique locations...


[tqrouting] Duration matrix ready (5×5, 0.0s).


[tqrouting] Solving VRP instance (4 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 2 routes, duration 4287.8, 0 unvisited).


[tqrouting] Computing duration matrix for 5 unique locations...


[tqrouting] Duration matrix ready (5×5, 0.0s).


[tqrouting] Solving VRP instance (4 customers, 1 vehicle types)...


fixed_vehicle_cost=0: 2 vehicle(s), total duration 4288s


[tqrouting] Solver finished in 2.0s (status=Feasible, 1 routes, duration 6382.8, 0 unvisited).


fixed_vehicle_cost=5000: 1 vehicle(s), total duration 6383s


## 4. Pricing waiting time

Arriving before a time window opens means waiting. The solver already shifts
each route's departure to cancel avoidable waiting; what remains is
*unavoidable* — and free by default. `penalty_waiting` prices each second of
it into the objective (it never affects feasibility).

Here `early` closes at 600 s and `late` opens at 4000 s — a forced gap.
A third customer can be served inside the gap, at the cost of extra driving:

In [7]:
durations_w = [
    #  D    early late  filler
    [   0,  300,  600,  900],
    [ 300,    0,  300,  800],
    [ 600,  300,    0,  800],
    [ 900,  800,  800,    0],
]
inst_wait = VRPInstance(
    customers=[
        Customer(id="early", location=Location(matrix_id=1),
                 time_window=TimeWindow(start=0, end=600)),
        Customer(id="late", location=Location(matrix_id=2),
                 time_window=TimeWindow(start=4000, end=4600)),
        Customer(id="filler", location=Location(matrix_id=3)),
    ],
    fleet=[VehicleType(start_location=Location(matrix_id=0), n_vehicles=1)],
    duration_matrix=durations_w,
)

for pw in (0, 5):
    sol_w = router.solve(inst_wait, penalty_waiting=pw, time_limit=2)
    order = [v.customer_id for r in sol_w.routes for v in r.customer_sequence]
    print(f"penalty_waiting={pw}: order {order}, "
          f"waiting {sol_w.total_waiting_time or 0:.0f}s, "
          f"route duration {sol_w.total_route_duration:.0f}s")

[tqrouting] Solving VRP instance (3 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Feasible, 1 routes, duration 4500.0, 0 unvisited).


[tqrouting] Solving VRP instance (3 customers, 1 vehicle types)...


penalty_waiting=0: order ['early', 'late', 'filler'], waiting 3100s, route duration 4500s


[tqrouting] Solver finished in 2.0s (status=Feasible, 1 routes, duration 3700.0, 0 unvisited).


penalty_waiting=5: order ['early', 'filler', 'late'], waiting 1800s, route duration 3700s


With free waiting the solver minimizes driving and idles 3100 s at `late`'s
door. Priced at 5 objective units per second, it spends 500 s of extra
driving to serve `filler` inside the gap and cuts the wait to 1800 s.

## 5. Soft constraints

A hard constraint makes violating solutions infeasible. Marking it soft prices
violations into the objective instead: the solve returns
`FeasibleWithSoftViolations` and reports each violation. The matching
`penalty_*` value is the price per unit of excess.

In [8]:
import warnings

# One slow vehicle, two far-apart customers with incompatible tight windows.
inst_soft = VRPInstance(
    customers=[
        Customer(id="west", location=Location(x=-3000, y=0),
                 time_window=TimeWindow(start=0, end=3300)),
        Customer(id="east", location=Location(x=3000, y=0),
                 time_window=TimeWindow(start=0, end=3300)),
    ],
    fleet=[VehicleType(start_location=Location(x=0, y=0), n_vehicles=1)],
)
inst_soft.compute_duration_matrix(EuclideanMatrixProvider())

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    sol_hard = router.solve(inst_soft, time_limit=2)
print(f"Hard windows: {sol_hard.solution_status}")
for w in caught:
    print(f"  warning: {w.message}")

[tqrouting] Computing duration matrix for 3 unique locations...


[tqrouting] Duration matrix ready (3×3, 0.0s).


[tqrouting] Solving VRP instance (2 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=Infeasible, 1 routes, duration 9000.0, 0 unvisited).


Hard windows: Infeasible


In [9]:
sol_soft = router.solve(
    inst_soft,
    soft_constraints=["time_window"],
    penalty_time_window=2.0,   # 2 objective units per second of lateness
    time_limit=2,
)
print(f"Soft windows: {sol_soft.solution_status}")
for v in sol_soft.constraint_violations:
    print(f"  {v.constraint}: customer {v.customer_id}, excess {v.excess_value:.0f}s")

[tqrouting] Solving VRP instance (2 customers, 1 vehicle types)...


[tqrouting] Solver finished in 2.0s (status=FeasibleWithSoftViolations, 1 routes, duration 9000.0, 0 unvisited).


Soft windows: FeasibleWithSoftViolations
  time_window: customer east, excess 5700s
